In [ ]:
# Packages not in the default online computer setup
%pip install tomli_w

In [ ]:
# This notebook shows how to take some shape information, format it to suit the echoSMs
# anatomical datastore format, add metadata, and write out in TOML format.

# This code can be a starting point for writing your own Python code to convert your shape
# data into anatomical datastore TOML files.

# This demo code only creates one specimen containing one shape.
# You may find it useful to generalise it to create multiple specimens with multiple shapes.

import uuid  # Used to generate universally unique identification strings
import tomli_w  # Used to write out TOML files
import requests  # To make web API calls
import numpy as np
from datetime import date
from pathlib import Path
from pprint import pprint  # to print dictionary data structures in a nicer format
from echosms import plot_specimen
import jsonschema_rs

schema_url = 'https://raw.githubusercontent.com/ices-tools-dev/echoSMs/refs/heads/main/data_store/schema/v1/anatomical_data_store.json'

# WoRMS is used to get species information
worms_url = 'https://www.marinespecies.org/rest/'

# A specimen template. Refer to the schema documentation for explanations of these, including
# data types, allowed values, and which are mandatory.

# needs to be unique across all datasets
dataset_uuid = str(uuid.uuid4())

specimen = {'uuid': str(uuid.uuid4()),  # The UUID for this specimen
           'dataset_uuid': dataset_uuid,
           'description': [""],
           'version_number': 1,
           'version_time': date.today().strftime('%Y-%m-%d'),
           'version_note': '',
           'anatomical_category': "organism",
           'aphia_id': 126436,  # this is Atlantic cod. Visit www.marinespecies.org to find the id for your species
           'date_collection': "",
           'date_image': "",
           'version_investigators': [],
           'data_collection_description': "",
           'notes': [""],
           'imaging_method': "unknown",
           'shape_method': "unknown",
           'shape_method_processing': "unknown",
           'model_type': "KRM",
           'sound_speed_method': "unknown",
           'mass_density_method': "unknown",
           'dataset_size': 0.0,  # the datatore loader program will overwrite this
           'dataset_size_units': 'megabyte',
           'specimen_name': 'Simulated specimen',
           'specimen_condition': 'unknown',
           'vernacular_names': [],
           'length': 0.45,
           'length_units': 'm',
           'sex': 'unknown',
           'length_type': 'unknown',
           'shape_type': '',  # set when the shape (see code below)
           'shapes': []}

In [ ]:
# Query WoRMS to get the full species classification naming using the aphia ID.
#  This is just a convenience as you could provide these attributes manually

r = requests.get(worms_url + 'AphiaRecordByAphiaID/' + str(specimen['aphia_id']))
if r.status_code != 200:
    print('Failed to get record for aphia ID of {specimen["aphia_id"]}')
aphia_data = r.json()

# and populate the relevant echoSMs anatomical datastore metadata fields
for attr in ['class', 'order', 'family', 'genus']:
    specimen[attr] = aphia_data[attr]
specimen['species'] = aphia_data['scientificname']

# Get the vernacular names from WoRMS too
r = requests.get(worms_url + 'AphiaVernacularsByAphiaID/' + str(specimen['aphia_id']))
# There can be many vernacular names. Add them all in.
if r.status_code == 200:
    for vname in r.json():
        specimen['vernacular_names'].append(vname['vernacular'])

# print the vernacular names to see what was provided by WoRMS
pprint(specimen['vernacular_names'])

In [ ]:
# Create a shape data structure with a simple shape.
# The fields here are for an outline shape.
shape1 = {'shape_units': 'm',
          'boundary': 'fluid-filled',
          'anatomical_feature': 'body',
          'x': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],  # ordered tail to head
          'y': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
          'z': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
          'height': [0.0, 0.1, 0.1, 0.1, 0.1, 0.0],  # the shape height at each (x,y,z) point
          'width': [0.0, 0.15, 0.15, 0.15, 0.15, 0.0],  # the shape width at each (x,y,z) point
          'mass_density': [970],
          'mass_density_units': 'kg/m^3',
          'sound_speed_compressional': [1450],
          'sound_speed_compressional_units': 'm/s'}

shape2 = {'shape_units': 'm',
          'boundary': 'pressure-release',
          'anatomical_feature': 'swimbladder',
          'x': [0.1, 0.2, 0.3, 0.4],  # ordered tail to head
          'y': [0.0, 0.0, 0.0, 0.0],
          'z': [0.0, 0.0, 0.0, 0.0],
          'height': [0.0, 0.05, 0.05, 0.0],  # the shape height at each (x,y,z) point
          'width': [0.0, 0.05, 0.05, 0.0],  # the shape width at each (x,y,z) point
          'mass_density': [970],
          'mass_density_units': 'kg/m^3',
          'sound_speed_compressional': [1450],
          'sound_speed_compressional_units': 'm/s'}

# Add the shape template into the specimen data structure
specimen['shapes'].append(shape1)
specimen['shapes'].append(shape2)
# And set the shape type
specimen['shape_type'] = 'outline'

In [ ]:
# Plot the shape using an echoSMs convenience function
plot_specimen(specimen)

In [ ]:
# Validate the specimen data structure

# Download the anatomical datastore schema from github
schema = requests.get(schema_url).json()

# Validate the TOML file against the schema
validator = jsonschema_rs.validator_for(schema)

# Use the validator in a way that reports which parts of the schema
# are not met.

# Can change an attribute to something invalid (for example, version number must be an integer)
# to see how errors are reported:
# specimen['version_number'] = 'A'

if not validator.is_valid(specimen):
    for error in validator.iter_errors(specimen):
        print(f'{error}')
else:
    print('Input data is valid.')

In [ ]:
# Write the entire dataset to a TOML file

# It might also be sensible to do the validatation against the schema here - the 
# datastore loading process does validation, but it's probably better to know about
# validation errors now rather than later.

# Convert any arrays to lists (to facilitate exporting to toml)
for s in specimen['shapes']:
    for k in s.keys():
        if isinstance(s[k], np.ndarray):
            s[k] = s[k].tolist()

# As per the echoSMs datatore guidelines, the TOML file should be placed in 
# a directory with the same name as the dataset_id
specimen_file = Path.home()/'datasets'/specimen['dataset_uuid']/'metadata.toml'

# Create the directory for the TOML file
Path.mkdir(specimen_file.parent, parents=True, exist_ok=True)

# Write the TOML file
with open(specimen_file, 'wb') as f:
    tomli_w.dump(specimen, f)
print(f'Wrote TOML file to {specimen_file}')